## This file does the following things:
1) link records with permission to linkage with Hospital administrative LSOA, for demographics
ID_to_link_with_demographics.csv

2) Cross-tabulate:
By year of birth, gender, ethnicity
Age at time of recording: date of birth - date of recording 



In [ ]:
import pandas as pd


In [ ]:
df_cross = pd.read_csv("cross_sectional_min_diff.csv")
df_lsoa_num = pd.read_csv("ID_to_link_with_demographics.csv")
df_demographic = pd.read_csv(r'S:\LLC_0002\lamj\notebooks_lsoa\LSOA\demographics_geo.csv')
df_nonhsd = pd.read_csv("no_nhsd_record.csv")

df_one_lsoa_cohort = pd.read_csv("one_LSOA_cohort.csv")
df_one_lsoa_nhsd = pd.read_csv("one_LSOA_NHSD.csv")


In [ ]:
# no restriction (include only 1 reported LSOA)
df_longitudinal = pd.read_csv("longitudinal_proportion_comparison_norestrict.csv", index_col = False)


In [ ]:
df_longitudinal

In [ ]:
df_cross

In [ ]:
df_lsoa_num[df_lsoa_num['num_different_lsoa'] == 1]

In [ ]:
# NHSD - 15,737
df_one_lsoa_nhsd

In [ ]:
# cohort. 16,878 - should use this for comparison.
df_one_lsoa_cohort

In [ ]:
df_demographic 

In [ ]:
columns = ["llc_0002_stud_id", "age", "ethnicity", "sex", "cohort"]
df_demographic = df_demographic[columns]

In [ ]:
df_demographic

In [ ]:
# save
df_demographic.to_csv(r"S:\LLC_0002\lamj\Datasets\Multiple-Source_Harmonisation\demographics.csv")


In [ ]:
# combine - demo + lsoa num
df_demo = pd.concat([df_lsoa_num, df_demographic],axis = 1, join = 'inner')



In [ ]:
df_demo

In [ ]:
columns_to_keep = ['llc_0002_stud_id', "num_different_lsoa","num_lsoa", "age", "ethnicity", "sex", "cohort"]
df_demo = df_demo[columns_to_keep]
df_demo = df_demo.loc[:, ~df_demo.columns.duplicated()]
df_demo

In [ ]:
df_longi = pd.concat([df_longitudinal, df_demo], axis = 1, join = 'inner')


In [ ]:
df_longi = df_longi.loc[:, ~df_longi.columns.duplicated()]

df_longi

In [ ]:
# link again, this time with cohort (one lsoa)
df_longi['no_change_lsoa_cohort'] = df_longi['llc_0002_stud_id'].isin(df_one_lsoa_cohort['llc_0002_stud_id'])
df_longi['no_change_lsoa_nhsd'] = df_longi['num_different_lsoa'] == 1
df_longi['lsoa11_e_cohort'] = df_one_lsoa_cohort['lsoa11_e']

In [ ]:
df_longi[df_longi['no_change_lsoa_cohort'] == True]['lsoa11_e_cohort'].value_counts()

In [ ]:
# cross tab to understand no changes demographics.

crosstab_nochange = pd.crosstab(df_longi['no_change_lsoa_cohort'], df_longi['no_change_lsoa_nhsd'], margins =True)
percentage_nochange = pd.crosstab(df_longi['no_change_lsoa_cohort'], df_longi['no_change_lsoa_nhsd'], normalize='all') * 100
percentage_nochange


In [ ]:
df_longi['num_lsoa'].describe()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# bin num_lsoa
#bins = np.linspace(df_longi['num_lsoa'].min(), df_longi['num_lsoa'].max(), num = 5)
bins = [1,2,29, 56, 105,df_longi['num_lsoa'].max()]
labels = [f'{int(bins[i])}-{int(bins[i+1]-1)}' for i in range(len(bins)-1)]
df_longi['bin'] = pd.cut(df_longi['num_lsoa'], bins = bins, labels = labels, include_lowest = True)

mean_values = df_longi.groupby('bin')[['overlap_prop_n_1', 'overlap_prop_median', 'overlap_prop_random']].mean()
sem_values = df_longi.groupby('bin')[['overlap_prop_n_1', 'overlap_prop_median', 'overlap_prop_random']].sem()

plt.figure(figsize = (10, 6))

plt.plot(mean_values.index, mean_values['overlap_prop_n_1'], label = "N-1", marker = 'o', color = 'skyblue')
plt.plot(mean_values.index, mean_values['overlap_prop_median'], label = "median", marker = 'o', color = 'salmon')
plt.plot(mean_values.index, mean_values['overlap_prop_random'], label = "random", marker = 'o', color = 'purple')

plt.xlabel("Binned - Number of LSOA")
plt.ylabel("Mean Proportion")
#plt.title('Proportion of shared encrypted LSOA in cohort and NHSD, by number of rows of available record')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# with error bars

bins = [1,2,29, 56, 105,df_longi['num_lsoa'].max()]
labels = [f'{int(bins[i])}-{int(bins[i+1]-1)}' for i in range(len(bins)-1)]
df_longi['bin'] = pd.cut(df_longi['num_lsoa'], bins = bins, labels = labels, include_lowest = True)

mean_values = df_longi.groupby('bin')[['overlap_prop_n_1', 'overlap_prop_median', 'overlap_prop_random']].mean()
sem_values = df_longi.groupby('bin')[['overlap_prop_n_1', 'overlap_prop_median', 'overlap_prop_random']].sem()

plt.figure(figsize = (10, 6))

plt.errorbar(mean_values.index, mean_values['overlap_prop_n_1'], yerr=sem_values['overlap_prop_n_1'], label = "N-1", marker = 'o', color = 'skyblue', capsize = 5)
plt.errorbar(mean_values.index, mean_values['overlap_prop_median'], yerr=sem_values['overlap_prop_median'],  label = "median", marker = 'o', color = 'salmon', capsize = 5)
plt.errorbar(mean_values.index, mean_values['overlap_prop_random'], yerr=sem_values['overlap_prop_random'],  label = "random", marker = 'o', color = 'purple', capsize = 5)


plt.xlabel("Binned - Number of reported LSOAs in NHSD")
plt.ylabel("Mean Proportion")
# plt.title('Proportion of shared encrypted LSOA in cohort and NHSD, by number of rows of available record')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
df_longi_change = df_longi[df_longi['no_change_lsoa_cohort'] != True]
df_longi_change['num_lsoa'].describe()

In [ ]:
# with error bars - limit to at least 1 change

bins = [1,2,29, 56, 105,df_longi['num_lsoa'].max()]
labels = [f'{int(bins[i])}-{int(bins[i+1]-1)}' for i in range(len(bins)-1)]
df_longi_change['bin'] = pd.cut(df_longi_change['num_lsoa'], bins = bins, labels = labels, include_lowest = True)


mean_values = df_longi_change.groupby('bin')[['overlap_prop_n_1', 'overlap_prop_median', 'overlap_prop_random']].mean()
sem_values = df_longi_change.groupby('bin')[['overlap_prop_n_1', 'overlap_prop_median', 'overlap_prop_random']].sem()
plt.figure(figsize = (10, 6))

plt.errorbar(mean_values.index, mean_values['overlap_prop_n_1'], yerr=sem_values['overlap_prop_n_1'], label = "N-1", marker = 'o', color = 'skyblue', capsize = 5)
plt.errorbar(mean_values.index, mean_values['overlap_prop_median'], yerr=sem_values['overlap_prop_median'],  label = "median", marker = 'o', color = 'salmon', capsize = 5)
plt.errorbar(mean_values.index, mean_values['overlap_prop_random'], yerr=sem_values['overlap_prop_random'],  label = "random", marker = 'o', color = 'purple', capsize = 5)


plt.xlabel("Binned - Number of reported LSOAs in NSHD")
plt.ylabel("Mean Proportion")
# plt.title('Proportion of shared encrypted LSOA in cohort and NHSD, by number of rows of available record')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# describe stats
df_longi_change.groupby('bin')[['overlap_prop_n_1', 'overlap_prop_median','overlap_prop_random']].mean()




In [ ]:
df_longi_nochange = df_longi[df_longi['no_change_lsoa_cohort'] == True]
# with error bars - limit to at least 1 change

bins = [1,2,29, 56, 105,df_longi['num_lsoa'].max()]
labels = [f'{int(bins[i])}-{int(bins[i+1]-1)}' for i in range(len(bins)-1)]
df_longi_nochange['bin'] = pd.cut(df_longi_nochange['num_lsoa'], bins = bins, labels = labels, include_lowest = True)


mean_values = df_longi_nochange.groupby('bin')[['overlap_prop_n_1', 'overlap_prop_median', 'overlap_prop_random']].mean()
sem_values = df_longi_nochange.groupby('bin')[['overlap_prop_n_1', 'overlap_prop_median', 'overlap_prop_random']].sem()
plt.figure(figsize = (10, 6))

plt.errorbar(mean_values.index, mean_values['overlap_prop_n_1'], yerr=sem_values['overlap_prop_n_1'], label = "N-1", marker = 'o', color = 'skyblue', capsize = 5)
plt.errorbar(mean_values.index, mean_values['overlap_prop_median'], yerr=sem_values['overlap_prop_median'],  label = "median", marker = 'o', color = 'salmon', capsize = 5)
plt.errorbar(mean_values.index, mean_values['overlap_prop_random'], yerr=sem_values['overlap_prop_random'],  label = "random", marker = 'o', color = 'purple', capsize = 5)


plt.xlabel("Binned - Number of reported LSOAs in NSHD")
plt.ylabel("Mean Proportion")
# plt.title('Proportion of shared encrypted LSOA in cohort and NHSD, by number of rows of available record')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()



In [ ]:
#df_longi.groupby('bin')[['overlap_prop_n_1', 'overlap_prop_median']].sem()
df_longi.groupby('bin')[['overlap_prop_n_1', 'overlap_prop_median','overlap_prop_random']].mean()

In [ ]:
df_longi_nochange.groupby('bin')[['overlap_prop_n_1', 'overlap_prop_median','overlap_prop_random']].mean()

In [ ]:
df_longi_change[['overlap_prop_n_1', 'overlap_prop_median','overlap_prop_random']].describe()


In [ ]:
df_longi[['overlap_prop_n_1', 'overlap_prop_median', 'overlap_prop_random']].sem()

In [ ]:
df_longi['cohort'].value_counts().sort_index()

In [ ]:
# cross_sectional

df_cro = pd.concat([df_cross, df_demo], axis = 1, join = 'inner')


In [ ]:
df_cro = df_cro.loc[:, ~df_cro.columns.duplicated()]

In [ ]:
df_cro[df_cro['num_different_lsoa'] != 1]

In [ ]:
df_cro['age'].value_counts()

In [ ]:
age_order = ['<=18 years','19-30 years','31-59 years','60-74 years','75+ years', 'Missing']
df_cro['age'] = pd.Categorical(df_cro['age'],categories = age_order, ordered = True)

In [ ]:
df_cro['min_date_diff_year'] = df_cro['min_date_diff']/365.25

In [ ]:
df_cro

In [ ]:
df_cro

In [ ]:
bins = [-100, -10, -5, -2, 2, 5, 10, 100]
labels = ['<-10 years', '-5 to -10 years', '-2 to -5 years', '-2 to 2 years', '2 to 5 years', '5 to 10 years', '> 10 years']
df_cro['min_date_diff_year_bin']= pd.cut(df_cro['min_date_diff_year'], bins = bins, labels = labels, right = False)
df_cro['age'] = df_cro['age'].fillna("Missing")
df_cro['min_date_diff_year_bin'].value_counts()

In [ ]:
#crosstab = pd.crosstab(df_cro['age'], df_cro['min_date_diff_year_bin'], margins = True, margins_name = 'Total')
crosstab = pd.crosstab(df_cro['age'], df_cro['min_date_diff_year_bin'])

crosstab


In [ ]:
"""percentage = pd.crosstab(df_cro['age'], df_cro['min_date_diff_year_bin'], margins = True, margins_name = 'Total', normalize = 'index') * 100
percentage
percentage.round(1).applymap(lambda x: f"{x}")"""

percentage = crosstab.div(crosstab.sum(axis=1), axis = 0) * 100
percentage

In [ ]:
combined = crosstab.applymap(str) + " (" + percentage.round(1).applymap(lambda x: f"{x}") + "%)"
combined

In [ ]:
# simple stacked plot
crosstab.plot(kind = 'bar', stacked = True, colormap = 'viridis', figsize = (10,6))

plt.xlabel('Age')
plt.ylabel('Count')
plt.legend(title = 'Min year diff by age', bbox_to_anchor = (1,1), loc = 'upper right')
plt.tight_layout()
plt.show()


In [ ]:
"""ax = crosstab.plot(kind = 'bar', stacked = True, colormap = 'viridis', figsize = (10,6))

for i, (age, row) in enumerate(percentage.iterrows()):
    cumulative = 0
    for j, min_date_diff_year_bin in enumerate(percentage.columns):
        pct_value = row[min_date_diff_year_bin]
        if pct_value > 0:
            ax.text(i, cumulative + pct_value *0.05, f'{pct_value:.1f}%',
                   ha ='center', va = 'bottom', color = 'white', fontsize = 10)
        cumulative += pct_value

plt.xlabel('Age')
plt.ylabel('Count')
plt.legend(title = 'Minimum year differences by age', bbox_to_anchor = (1.05,1), loc = 'upper right')
plt.tight_layout()
plt.show()"""

In [ ]:
df_cro_nochange = df_cro[df_cro['num_different_lsoa'] == 1]
crosstab2 = pd.crosstab(df_cro_nochange['age'], df_cro_nochange['min_date_diff_year_bin'])
crosstab2

In [ ]:
crosstab2.plot(kind = 'bar', stacked = True, colormap = 'viridis', figsize = (10,6))

plt.xlabel('Age')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Plot: variation from zero - raincloud plot using only seaborn should be fine.
import seaborn as sns

p25 = np.percentile(df_cro['min_date_diff'], 25)
p50 = np.percentile(df_cro['min_date_diff'], 50)
p75 = np.percentile(df_cro['min_date_diff'], 75)

plt.figure(figsize = (10,6))

palette = sns.color_palette("Set2", n_colors = len(df_cro['age'].unique()))
unique_categories = ['<=18 years','19-30 years','31-59 years', '60-74 years', '75+ years', 'Missing']
color_map = dict(zip(unique_categories, palette))
colors = df_cro['age'].map(color_map)

sns.violinplot(x = 'min_date_diff', data=df_cro, inner = None, color = "White", bw= 0.3, zorder = 2)

sns.boxplot(x ='min_date_diff', data=df_cro, whis = (5,95), width = 0.2, color = 'lightblue', zorder = 3)

y_jittered = np.random.uniform(0.7, 0.08, size = len(df_cro))
plt.scatter(df_cro['min_date_diff'], y_jittered, alpha = 0.2, c = colors, edgecolor = 'none', zorder= 4)

"""
for i, row in df_cro.iterrows():
    plt.scatter(row['min_date_diff'], y_jittered[i],
               alpha = 0.6,
               color=category_to_colour[[age_cat]],
               edgecolor='black', zorder= 4)
"""

plt.annotate('-2689 (P25)', xy = (p25, 0), xytext =(p25, -0.08), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.annotate('-743 (P50)', xy = (p50, 0), xytext =(p50, -0.15), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.annotate('-243 (P75)', xy = (p75, 0), xytext =(p75, -0.08), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')

plt.title('Any Match: Min Date Difference')
plt.xlabel('Date Difference (days)')
plt.grid(True)
plt.yticks([])

#handles = [plt.Line2D([0], [0], marker = 'o', color = color, label = age, markersize= 10) for age, color in color_map.items()]
#plt.legend(handles=handles, title='Age Category', loc = 'upper right')

plt.show()

In [ ]:
# Plot: in years
import seaborn as sns

df_cro['min_date_diff_year'] = df_cro['min_date_diff']/365.25

p25 = np.percentile(df_cro['min_date_diff_year'], 25)
p50 = np.percentile(df_cro['min_date_diff_year'], 50)
p75 = np.percentile(df_cro['min_date_diff_year'], 75)

plt.figure(figsize = (10,6))

"""palette = sns.color_palette("Set2", n_colors = len(df_cro['age'].unique()))
unique_categories = ['<=18 years','19-30 years','31-59 years', '60-74 years', '75+ years', 'Missing']
color_map = dict(zip(unique_categories, palette))
colors = df_cro['age'].map(color_map)"""

sns.violinplot(x = 'min_date_diff_year', data=df_cro, inner = None, color = "White", bw= 0.3, zorder = 2)

sns.boxplot(x ='min_date_diff_year', data=df_cro, whis = (5,95), width = 0.2, showfliers = False, color = 'lightblue', zorder = 3)

"""y_jittered = np.random.uniform(0.5, 0.05, size = len(df_cro))
plt.scatter(df_cro['min_date_diff_year'], y_jittered, alpha = 0.4, c = colors, edgecolor = 'none', zorder= 4)"""

plt.annotate('-7.4 (P25)', xy = (p25, 0), xytext =(p25, -0.08), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.annotate('-2.0 (P50)', xy = (p50, 0), xytext =(p50, -0.15), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.annotate('-0.7 (P75)', xy = (p75, 0), xytext =(p75, -0.08), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')

plt.title('Any Match: Min Start Date Difference')
plt.xlabel('Date Difference (years)')
plt.grid(True)
plt.yticks([])

"""handles = [plt.Line2D([0], [0], marker = 'o', color = color, label = age, markersize= 10) for age, color in color_map.items()]


plt.legend(handles=handles, title='Age Category', loc = 'upper right')"""

plt.show()

In [ ]:
# Same plot but restrict to born after 1995

df_cro_1995 = df_cro_LPS[df_cro_LPS['age'] != "19-30 years"]
df_cro_1995


df_cro_1995['min_date_diff_year'] = df_cro_1995['min_date_diff']/365.25

p25 = np.percentile(df_cro_1995['min_date_diff_year'], 25)
p50 = np.percentile(df_cro_1995['min_date_diff_year'], 50)
p75 = np.percentile(df_cro_1995['min_date_diff_year'], 75)




In [ ]:
plt.figure(figsize = (10,6))

"""palette = sns.color_palette("Set2", n_colors = len(df_cro['age'].unique()))
unique_categories = ['<=18 years','19-30 years','31-59 years', '60-74 years', '75+ years', 'Missing']
color_map = dict(zip(unique_categories, palette))
colors = df_cro['age'].map(color_map)"""

sns.violinplot(x = 'min_date_diff_year', data=df_cro_1995, inner = None, color = "White", bw= 0.3, zorder = 2)

sns.boxplot(x ='min_date_diff_year', data=df_cro_1995, whis = (5,95), width = 0.2, showfliers = False, color = 'lightblue', zorder = 3)

"""y_jittered = np.random.uniform(0.5, 0.05, size = len(df_cro))
plt.scatter(df_cro['min_date_diff_year'], y_jittered, alpha = 0.4, c = colors, edgecolor = 'none', zorder= 4)"""

plt.annotate('-7.4 (P25)', xy = (p25, 0), xytext =(p25, -0.08), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.annotate('-2.0 (P50)', xy = (p50, 0), xytext =(p50, -0.15), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.annotate('-0.7 (P75)', xy = (p75, 0), xytext =(p75, -0.08), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')

plt.title('Any Match: Min Start Date Difference')
plt.xlabel('Date Difference (years)')
plt.grid(True)
plt.yticks([])

"""handles = [plt.Line2D([0], [0], marker = 'o', color = color, label = age, markersize= 10) for age, color in color_map.items()]


plt.legend(handles=handles, title='Age Category', loc = 'upper right')"""

plt.show()


In [ ]:
# Plot: in years
import seaborn as sns

df_cro_LPS['min_date_diff_year'] = df_cro_LPS['min_date_diff']/365.25

p25 = np.percentile(df_cro_LPS['min_date_diff_year'], 25)
p50 = np.percentile(df_cro_LPS['min_date_diff_year'], 50)
p75 = np.percentile(df_cro_LPS['min_date_diff_year'], 75)


In [ ]:
df_cro_LPS['age']

In [ ]:
sns.set(style = "whitegrid")

plt.figure(figsize = (10,6))
"""palette = sns.color_palette("Set2", n_colors = len(df_cro['age'].unique()))
unique_categories = ['<=18 years','19-30 years','31-59 years', '60-74 years', '75+ years', 'Missing']
color_map = dict(zip(unique_categories, palette))
colors = df_cro['age'].map(color_map)"""

ax = sns.violinplot(
    x = "age",
    y = "min_date_diff_year",
    hue = "age",
    data= df_cro_LPS,
    inner = None,
    palette= "Set2",
    bw = 0.1,
    zorder = 2,
)
#x = 'min_date_diff_year', data=df_cro, inner = None, color = "White", bw= 0.3, zorder = 2)
sns.boxplot(
    x = "age",
    y = 'min_date_diff_year',
    hue = "age",
    data=df_cro_LPS, 
    whis = (5,95), 
    width = 0.2, 
    showfliers = False, 
    palette = 'Set2', 
    zorder = 2,
    dodge = True

)

"""y_jittered = np.random.uniform(0.5, 0.05, size = len(df_cro))
plt.scatter(df_cro['min_date_diff_year'], y_jittered, alpha = 0.4, c = colors, edgecolor = 'none', zorder= 4)"""

"""plt.annotate('-7.4 (P25)', xy = (p25, 0), xytext =(p25, -0.08), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.annotate('-2.0 (P50)', xy = (p50, 0), xytext =(p50, -0.15), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
plt.annotate('-0.7 (P75)', xy = (p75, 0), xytext =(p75, -0.08), arrowprops= dict(arrowstyle='-', color = 'black'), fontsize = 10, color = 'black', ha= 'center')
"""
#plt.title('Any Match: Min Start Date Difference')
handles, labels = ax.get_legend_handles_labels()

by_label = dict(zip(labels, handles))
ax.legend(by_label.values(), by_label.keys(), title = "Age Group")

#plt.xlabel('Date Difference (years)')
plt.grid(True)
#plt.yticks([])
#plt.legend(title = "Age", loc = "upper right")

"""handles = [plt.Line2D([0], [0], marker = 'o', color = color, label = age, markersize= 10) for age, color in color_map.items()]


plt.legend(handles=handles, title='Age Category', loc = 'upper right')"""

plt.show()

In [ ]:
df_cro['min_date_diff_year'].describe()

In [ ]:
df_cro['sex'].value_counts(dropna =False)
df_cro['sex'] = df_cro['sex'].fillna("Missing")

In [ ]:
df_cro['cohort'].value_counts(dropna=False)

In [ ]:
df_cro['lsoa_cohort'].value_counts()

In [ ]:
df_cro['age'].value_counts(dropna =False)
df_cro['age'] = df_cro['age'].fillna("Missing")

In [ ]:
df_cro['ethnicity'].value_counts(dropna =False)
df_cro['ethnicity'] = df_cro['ethnicity'].fillna("Missing")

In [ ]:
### NEW DEMOGRAPHICS TABLE
import pandas as pd
from tableone import TableOne
columns = ['sex', 'ethnicity', 'age']
categorical = ['sex','ethnicity', 'age']
order = {
    "age":["<=18 years","19-30 years","31-59 years","60-74 years","75+ years", "Missing"],
    "ethnicity": ["White", "South-east Asian", "Other Asian", "Black", "Mixed", "Other", "Missing"],
    "sex":["Female", "Male", "Missing"]
}

mytable = TableOne(df_cro, columns, categorical,order = order)
mytable.tableone




In [ ]:
df_cro_nochange = df_cro[df_cro['num_different_lsoa'] == 1]
df_cro_nochange['lsoa_cohort'].value_counts()

In [ ]:
# no change
mytable_nochg = TableOne(df_cro_nochange, columns, categorical,order = order)
mytable_nochg.tableone

In [ ]:
# plot demographic tables - None, cross-sectional, Longitudinal 

In [ ]:
# plot No LSOA change
from tableone import TableOne


# table 1, no change vs no lsoa 
columns = ['sex', 'ethnicity', 'age']
categorical = ['sex','ethnicity', 'age']
order = {
    "age":["<=18 years","19-30 years","31-59 years","60-74 years","75+ years", "Missing"],
    "ethnicity": ["White", "South-east Asian", "Other Asian", "Black", "Mixed", "Other", "Missing"],
    "sex":["Female", "Male", "Missing"]
}

df_longi_nochange['age'] = df_longi_nochange['age'].fillna("Missing")
df_longi_nochange['ethnicity'] = df_longi_nochange['ethnicity'].fillna("Missing")
df_longi_nochange['sex'] = df_longi_nochange['sex'].fillna("Missing")


nochange_cohort = TableOne(df_longi_nochange, columns, categorical,order = order)
nochange_cohort.tableone




In [ ]:
df_longi_nochange.columns

In [ ]:
df_nhsd_no = pd.concat([df_nonhsd, df_demo], axis = 1, join = 'inner')
df_nhsd_no = df_nhsd_no.loc[:, ~df_nhsd_no.columns.duplicated()]

df_cro['group'] = 'Linked'
df_nhsd_no['group'] = 'Unlinked'
df_combined = pd.concat([df_cro, df_nhsd_no], ignore_index = True)

In [ ]:
df_combined

In [ ]:
age_order = ['<=18 years', '19-30 years', '31-59 years', '60-74 years','75+ years', 'Missing']
sex_order = ['Female', 'Male', 'Missing', 'NA']
ethnicity_order = ['South-east Asian', 'Other Asian', 'Black', 'Mixed', 'White', 'Other', 'Missing', 'NA']

df_combined['age'] = pd.Categorical(df_combined['age'], categories = age_order, ordered = True)
df_combined['sex'] = pd.Categorical(df_combined['sex'], categories = sex_order, ordered = True)
df_combined['ethnicity'] = pd.Categorical(df_combined['ethnicity'], categories = ethnicity_order, ordered = True)


In [ ]:
def create_table1(df, vars_to_summarize, group_col = None):
    table = []

    for var in vars_to_summarize:
        if group_col:
            counts = (
            df.groupby(group_col)[var]
                .value_counts(normalize=False)
                .unstack(fill_value=0)
                .T
            )
            percentages = (
            df.groupby(group_col)[var]
                .value_counts(normalize=True)
                .unstack(fill_value=0)
                .T * 100
            )
        else:
            counts = df[var].value_counts(normalize = False).to_frame('count').T
            percentages = df[var].value_counts(normalize = True).to_frame('percentage').T * 100
            counts.index = ['Overall']
            percentages.index = ['Overall']
        
        
        # combine counts and percentages
        if group_col:
            summary = counts.astype(str) + " (" + percentages.round(1).astype(str) + "%)"
        else:
            summary = counts.astype(str) + " (" + percentages.round(1).astype(str) + "%)"
            summary = summary.T
        
        # Add level to index as heading
        summary.index = pd.MultiIndex.from_tuples(
            [(var, idx) for idx in summary.index],
            names = ["Variable", "Category"]
        )
        table.append(summary)
    return pd.concat(table)



In [ ]:
df_combined['num_different_lsoa'].describe()

In [ ]:
bins = [1,2,4,10,df_combined['num_different_lsoa'].max()+1]
labels = [f'{int(bins[i])}-{int(bins[i+1]-1)}' for i in range(len(bins)-1)]
df_combined['diff_bin'] = pd.cut(df_combined['num_different_lsoa'], bins = bins, labels = labels, include_lowest = True)
df_combined['diff_bin'] 

In [ ]:
vars_to_summarize = ['age','sex','ethnicity', 'diff_bin']

demographic_table = create_table1(df_combined, vars_to_summarize, 'group')

demographic_table

In [ ]:
df_longi['num_different_lsoa'].describe()
bins = [1,2,5,12,df_longi['num_different_lsoa'].max()+1]
labels = [f'{int(bins[i])}-{int(bins[i+1]-1)}' for i in range(len(bins)-1)]
df_longi['diff_bin'] = pd.cut(df_longi['num_different_lsoa'], bins = bins, labels = labels, include_lowest = True)
df_longi['diff_bin'] 

In [ ]:
vars_to_summarize = ['age','sex','ethnicity','diff_bin']

longi = create_table1(df_longi, vars_to_summarize)
longi

In [ ]:
combined_table = pd.concat([longi, demographic_table], keys = ["longitudinal", "cross-sectional", "not in NHSD"],axis = 1)


combined_table

In [ ]:
combined_table.to_csv("table1_demographics_v3.csv")

In [ ]:
len(df_cross['lsoa_cohort'].unique())